# HACKING

Hacking is gaining unauthorized access to a system. The there is three types. Gray hat is when you do it illegally but let a person know. You can guess white and black.

Whats BeEF?

A browser exploitation framework that allows you to use a number of exploitations against a target system.

What is Bettercap?

Allows you to injects beefs code into a brower. You can use spoof.code to

## Penetration Testing.

Modern pentesting involves running many commands across distinct phases of an assessment. This notebook uses a category‑driven scripting model to make command execution structured and digestible.

# 1. Prerequisites

All command blocks depend on a single helper function, run, which executes a command, captures a short preview, and prints a formatted table with a human‑readable note.

If this function is not defined before any category block, all subsequent cells will fail.

### 1.1. Required: `run` Helper

The following run function is responsible for running each command, capturing a short preview, printing a formatted table, attaching a human‑readable note. This ensures every subsequetion codeblock produces consistent, human‑digestible output. It is therefore required.

In [ ]:
%%bash

run() {
    output=$(eval "$1" 2>&1 | head -n 3)
    printf "+----------------------+----------------------------------------------+\n"
    printf "| COMMAND             | %-44s |\n" "$1"
    printf "+----------------------+----------------------------------------------+\n"
    printf "| RESULT              | %-44s |\n" "$output"
    printf "+----------------------+----------------------------------------------+\n"
    printf "| NOTES               | %-44s |\n" "$2"
    printf "+----------------------+----------------------------------------------+\n\n"
}

### 1.1.2. Tool Requirements

The following lists the required tools.

```
tools=(
  whois dig sublist3r whatweb
  ping netdiscover nmap
  curl dirb wfuzz
  cewl hydra john hashcat
  airmon-ng airodump-ng wash aireplay-ng
  uname ps sudo find netstat ss
)
```

To check the tools you can run the following:

```
printf "%-20s %-10s\n" "Tool" "Status"
printf "%-20s %-10s\n" "----" "------"

for t in "${tools[@]}"; do
    if command -v "$t" >/dev/null 2>&1; then
        printf "%-20s %-10s\n" "$t" "OK"
    else
        printf "%-20s %-10s\n" "$t" "MISSING"
    fi
done
```

If a tool is missing, the table still renders — the preview simply shows an error.

### 1.1.3.  Wireless Adapter

A wireless adapter with monitor mode, packet injection, and AP is needed for WEP and WPA/WPA2 cracking. Should support Kali linux. (Recommended chipsets include the RealTek TTL8812AU and the Atheros AR9271).

Each interface has a MAC address, a permanent, physical, unique id assigned by manufacturer. In hacking, the MAC is modified for anonymity and impersonating.
- managed: only capture packets that have the destination MAC

** Unplug adapter before connecting Kali Linux.

** Need to disable the interface to modify it



# 2. Scripting

This section defines the notebook’s scripting workflow. Each phase uses the run helper to execute commands, capture a short preview, and present results in clean tables.

The subsections follow this pattern across all major pentest phases:

| **Pentest Phase** | **Description** |
| --- | --- |
| **[Reconnaissance](ca://s?q=Explain_Reconnaissance_Category)** | Public information gathering |
| **[Network Scanning](ca://s?q=Explain_Network_Scanning_Category)** | Host and service discovery |
| **[Web Mapping](ca://s?q=Explain_Web_Application_Mapping_Category)** | Endpoint enumeration |
| **[Authentication](ca://s?q=Explain_Authentication_Testing_Category)** | Credential evaluation |
| **[Wi‑Fi Assessment](ca://s?q=Explain_WiFi_Assessment_Category)** | Wireless analysis |
| **[System Enumeration](ca://s?q=Explain_System_Enumeration_Category)** | OS and privilege inspection |

Each script applies the same execution pattern to maintain clarity across the notebook.

## 2.1. Reconnaissance Script

Recon is the first phase of an assessment, focused on gathering publicly available information before any active scanning. The following commands provide a quick profile of the target’s public‑facing infrastructure.

In [ ]:
%%bash

# Load environment variable for target domain
TARGET_DOMAIN="${TARGET_DOMAIN:-example.com}"
TARGET_URL="https://${TARGET_DOMAIN}"

# Domain ownership
run "whois ${TARGET_DOMAIN}" "Domain ownership lookup"

# DNS record enumeration
run "dig ${TARGET_DOMAIN} ANY" "DNS record enumeration"

# Passive subdomain discovery
run "sublist3r -d ${TARGET_DOMAIN}" "Subdomain discovery"

# Technology fingerprinting
run "whatweb ${TARGET_URL}" "Tech fingerprinting"

## 2.2. Network Scanning Script

Network scanning is the first active phase of an assessment, where we begin interacting with the target’s infrastructure to understand what services are exposed and how they behave.

In [ ]:
%%bash

# Load environment variables for network targets
GATEWAY_IP="${GATEWAY_IP:-192.168.1.1}"
NETWORK_RANGE="${NETWORK_RANGE:-192.168.1.0/24}"
TARGET_HOST="${TARGET_HOST:-192.168.1.10}"

# Ping the gateway to confirm it is reachable and responding
run "ping -c 4 ${GATEWAY_IP}" "Host reachability"

# Discover active hosts on the local network using ARP scanning
run "netdiscover -r ${NETWORK_RANGE}" "LAN host discovery"

# Scan a specific host to identify open ports, running services, and OS fingerprints
run "nmap -sV -O ${TARGET_HOST}" "Service + OS detection"

# Perform a full port scan on the target to enumerate all open ports
run "nmap -p- ${TARGET_HOST}" "Full port scan"


One important concept is packet sniffing. Packet sniffing allows you to capture all of the information about the network. E.g. signatures get repeated on active networks.

In [ ]:
%%bash

airmon-ng mon0
airodump-ng --band abg mon0
airodump-ng —-bssid ne:wm:ac:ad:dr:es -—channel 2 -—write test mon0

## 2.3. Web Application Mapping Script

Web mapping is the phase where we actively interact with a web application to understand its structure, exposed endpoints, and underlying technologies. It helps reveal headers, hidden directories, fuzzable parameters, and framework details that define the application’s attack surface.

In [ ]:
%%bash

# Load environment variables for target domain and URL
TARGET_DOMAIN="${TARGET_DOMAIN:-example.com}"
TARGET_URL="https://${TARGET_DOMAIN}"

# Retrieve only the HTTP response headers to inspect server metadata
run "curl -I ${TARGET_URL}" "Fetch headers"

# Enumerate hidden directories and files using a common wordlist
run "dirb ${TARGET_URL} /usr/share/wordlists/dirb/common.txt" "Directory brute force"

# Fuzz the 'param' parameter to discover hidden values or behaviors
run "wfuzz -c -w wordlist.txt -u ${TARGET_URL}/page.php?param=FUZZ" "Parameter fuzzing"

## 2.4. Authentication & Password Testing Script

Authentication testing evaluates how well a system resists unauthorized access by analyzing credential strength, wordlists, and login security. This phase covers both online authentication robustness and offline password hygiene, using the run helper to present each command in clean, readable tables instead of raw cracking output.

These commands form the foundation of credential security analysis, revealing weak passwords, vulnerable login endpoints, and hash quality. The cell below runs the authentication and password testing commands.

In [ ]:
%%bash

# Load environment variables for target URL, SSH host, and file paths
TARGET_DOMAIN="${TARGET_DOMAIN:-example.com}"
TARGET_URL="https://${TARGET_DOMAIN}"

SSH_HOST="${SSH_HOST:-192.168.1.10}"

USER_LIST="${USER_LIST:-users.txt}"
PASS_LIST="${PASS_LIST:-passwords.txt}"
HASH_FILE="${HASH_FILE:-hashes.txt}"
WORDLIST="${WORDLIST:-custom.txt}"

# Crawl the target site and generate a custom wordlist from its content
run "cewl ${TARGET_URL} -w ${WORDLIST}" "Generate wordlist"

# Test SSH login resilience using user and password lists
run "hydra -L ${USER_LIST} -P ${PASS_LIST} ssh://${SSH_HOST}" "SSH login testing"

# Perform offline password cracking against collected hashes
run "john ${HASH_FILE}" "Offline password cracking"

## 2.5. Wifi-Assessment Script

Capturing Wireless Metadata and Evaluating Security Posture
This section runs a set of Wi‑Fi assessment commands using the notebook’s table‑driven output format. Wireless testing often produces noisy terminal output, so the run helper keeps results concise and readable.

These commands form the foundation of wireless reconnaissance and security evaluation.

In [ ]:
%%bash

# Load environment variables for wireless interface and monitor interface
WIFI_IFACE="${WIFI_IFACE:-wlan0}"
MON_IFACE="${MON_IFACE:-${WIFI_IFACE}mon}"

# Enable monitor mode on the wireless interface for packet capture
run "airmon-ng start ${WIFI_IFACE}" "Enable monitor mode"

# Capture nearby Wi‑Fi traffic and display AP/client metadata
run "airodump-ng ${MON_IFACE}" "Packet capture"

# Check WPS status on visible access points
run "wash -i ${MON_IFACE}" "WPS check"

# Test whether the adapter supports packet injection
run "aireplay-ng --test ${MON_IFACE}" "Injection test"

## 2.6. System Enumeration Script

System enumeration is the phase where we inspect OS details, running processes, privileges, and local services to understand the system’s security posture and identify potential escalation paths. The cell below runs the enumeration commands and prints their results in table format.

In [ ]:
%%bash

# Load environment variables for optional overrides
TARGET_USER="${TARGET_USER:-$(whoami)}"
SUID_SEARCH_PATH="${SUID_SEARCH_PATH:-/}"

# Display kernel and OS details for baseline system identification
run "uname -a" "Kernel info"

# List all running processes to identify services and potential escalation targets
run "ps aux" "Process list"

# Show allowed sudo privileges for the current user
run "sudo -l" "Privilege escalation check"

# Enumerate SUID binaries that may provide escalation vectors
run "find ${SUID_SEARCH_PATH} -perm -4000 2>/dev/null" "SUID binaries"

# List listening network services and associated processes
run "netstat -tulnp" "Listening services"


# 3. Full Pentest

In [ ]:
%%bash

############################################################
# CONFIGURATION (Defaults)
############################################################

DOMAIN="${DOMAIN:-example.com}"
TARGET_IP="${TARGET_IP:-}"
SUBNET="${SUBNET:-}"
IFACE="${IFACE:-wlan0}"
WORDLIST="${WORDLIST:-wordlist.txt}"
HASHES="${HASHES:-hashes.txt}"
OUTPUT_FILE="${OUTPUT_FILE:-}"

############################################################
# ARGUMENT PARSER
############################################################

while [[ $# -gt 0 ]]; do
    case "$1" in
        --domain) DOMAIN="$2"; shift ;;
        --ip) TARGET_IP="$2"; shift ;;
        --subnet) SUBNET="$2"; shift ;;
        --iface) IFACE="$2"; shift ;;
        --wordlist) WORDLIST="$2"; shift ;;
        --hashes) HASHES="$2"; shift ;;
        --output) OUTPUT_FILE="$2"; shift ;;
        *)
            echo "Unknown argument: $1"
            exit 1
            ;;
    esac
    shift
done

############################################################
# VALIDATION
############################################################

[[ -z "$TARGET_IP" ]] && { echo "ERROR: --ip required"; exit 1; }
[[ -z "$SUBNET" ]] && { echo "ERROR: --subnet required"; exit 1; }

############################################################
# SAFE COMMAND WRAPPER
############################################################

run_cmd() {
    local label="$1"
    local cmd="$2"
    local tool
    tool=$(echo "$cmd" | awk '{print $1}')
    local timestamp=$(date -Iseconds)

    if ! command -v "$tool" >/dev/null 2>&1; then
        echo "\"$label\": {\"command\": \"$cmd\", \"output\": \"TOOL NOT INSTALLED\", \"status\": \"MISSING\", \"exit_code\": 127, \"timestamp\": \"$timestamp\"}"
        return
    fi

    # SAFE PLACEHOLDER — replace privately
    local output="placeholder output for $cmd"
    local exit_code=0
    local status="SUCCESS"

    echo "\"$label\": {\"command\": \"$cmd\", \"output\": \"$output\", \"status\": \"$status\", \"exit_code\": $exit_code, \"timestamp\": \"$timestamp\"}"
}

############################################################
# MODULES
############################################################

reconnaissance() {
    run_cmd "public_information_lookup" "whois $DOMAIN"
    run_cmd "dns_record_enumeration" "dig $DOMAIN ANY"
    run_cmd "subdomain_enumeration" "sublist3r -d $DOMAIN"
    run_cmd "web_technology_fingerprinting" "whatweb https://$DOMAIN"
}

network_scanning() {
    run_cmd "host_reachability_test" "ping -c 4 $TARGET_IP"
    run_cmd "lan_host_discovery" "netdiscover -r $SUBNET"
    run_cmd "service_and_os_detection" "nmap -sV -O $TARGET_IP"
    run_cmd "full_port_scan" "nmap -p- $TARGET_IP"
}

web_mapping() {
    run_cmd "fetch_http_headers" "curl -I https://$DOMAIN"
    run_cmd "directory_bruteforce" "dirb https://$DOMAIN /usr/share/wordlists/dirb/common.txt"
    run_cmd "parameter_fuzzing" "wfuzz -c -w $WORDLIST -u https://$DOMAIN/page.php?param=FUZZ"
}

authentication() {
    run_cmd "generate_custom_wordlist" "cewl https://$DOMAIN -w custom.txt"
    run_cmd "ssh_login_testing" "hydra -L users.txt -P passwords.txt ssh://$TARGET_IP"
    run_cmd "offline_hash_cracking" "john $HASHES"
}

wifi_assessment() {
    run_cmd "enable_monitor_mode" "airmon-ng start $IFACE"
    run_cmd "packet_capture" "airodump-ng ${IFACE}mon"
    run_cmd "wps_status_check" "wash -i ${IFACE}mon"
    run_cmd "packet_injection_test" "aireplay-ng --test ${IFACE}mon"
}

system_enumeration() {
    run_cmd "kernel_and_os_details" "uname -a"
    run_cmd "process_listing" "ps aux"
    run_cmd "sudo_privilege_check" "sudo -l"
    run_cmd "suid_binary_enumeration" "find / -perm -4000 2>/dev/null"
    run_cmd "listening_services" "netstat -tulnp"
}

############################################################
# JSON CATEGORY EMITTER
############################################################

emit_category() {
    local name="$1"
    shift
    local items=("$@")
    echo "\"$name\": {"
    local count=${#items[@]}
    for ((i=0; i<count; i++)); do
        echo "${items[$i]}"
        [[ $i -lt $((count-1)) ]] && echo ","
    done
    echo "}"
}

############################################################
# RUN MODULES
############################################################

mapfile -t RECON < <(reconnaissance)
mapfile -t NETSCAN < <(network_scanning)
mapfile -t WEBMAP < <(web_mapping)
mapfile -t AUTH < <(authentication)
mapfile -t WIFI < <(wifi_assessment)
mapfile -t SYSENUM < <(system_enumeration)

############################################################
# FINAL JSON
############################################################

FINAL_JSON=$(cat <<EOF
{
$(emit_category "reconnaissance" "${RECON[@]}"),
$(emit_category "network_scanning" "${NETSCAN[@]}"),
$(emit_category "web_mapping" "${WEBMAP[@]}"),
$(emit_category "authentication" "${AUTH[@]}"),
$(emit_category "wifi_assessment" "${WIFI[@]}"),
$(emit_category "system_enumeration" "${SYSENUM[@]}")
}
EOF
)

############################################################
# OUTPUT
############################################################

if [[ -n "$OUTPUT_FILE" ]]; then
    echo "$FINAL_JSON" > "$OUTPUT_FILE"
    echo "Saved JSON to $OUTPUT_FILE"
else
    echo "$FINAL_JSON"
fi


```
{
  "reconnaissance": {
    "public_information_lookup": {
      "command": "whois example.com",
      "output": "",
      "status": ""
    },
    "dns_record_enumeration": {
      "command": "dig example.com ANY",
      "output": "",
      "status": ""
    },
    "subdomain_enumeration": {
      "command": "sublist3r -d example.com",
      "output": "",
      "status": ""
    },
    "web_technology_fingerprinting": {
      "command": "whatweb https://example.com",
      "output": "",
      "status": ""
    }
  },

  "network_scanning": {
    "host_reachability_test": {
      "command": "ping -c 4 192.168.1.1",
      "output": "",
      "status": ""
    },
    "lan_host_discovery": {
      "command": "netdiscover -r 192.168.1.0/24",
      "output": "",
      "status": ""
    },
    "service_and_os_detection": {
      "command": "nmap -sV -O 192.168.1.10",
      "output": "",
      "status": ""
    },
    "full_port_scan": {
      "command": "nmap -p- 192.168.1.10",
      "output": "",
      "status": ""
    }
  },

  "web_mapping": {
    "fetch_http_headers": {
      "command": "curl -I https://example.com",
      "output": "",
      "status": ""
    },
    "directory_bruteforce": {
      "command": "dirb https://example.com /usr/share/wordlists/dirb/common.txt",
      "output": "",
      "status": ""
    },
    "parameter_fuzzing": {
      "command": "wfuzz -c -w wordlist.txt -u https://example.com/page.php?param=FUZZ",
      "output": "",
      "status": ""
    }
  },

  "authentication": {
    "generate_custom_wordlist": {
      "command": "cewl https://example.com -w custom.txt",
      "output": "",
      "status": ""
    },
    "ssh_login_testing": {
      "command": "hydra -L users.txt -P passwords.txt ssh://192.168.1.10",
      "output": "",
      "status": ""
    },
    "offline_hash_cracking": {
      "command": "john hashes.txt",
      "output": "",
      "status": ""
    }
  },

  "wifi_assessment": {
    "enable_monitor_mode": {
      "command": "airmon-ng start wlan0",
      "output": "",
      "status": ""
    },
    "packet_capture": {
      "command": "airodump-ng wlan0mon",
      "output": "",
      "status": ""
    },
    "wps_status_check": {
      "command": "wash -i wlan0mon",
      "output": "",
      "status": ""
    },
    "packet_injection_test": {
      "command": "aireplay-ng --test wlan0mon",
      "output": "",
      "status": ""
    }
  },

  "system_enumeration": {
    "kernel_and_os_details": {
      "command": "uname -a",
      "output": "",
      "status": ""
    },
    "process_listing": {
      "command": "ps aux",
      "output": "",
      "status": ""
    },
    "sudo_privilege_check": {
      "command": "sudo -l",
      "output": "",
      "status": ""
    },
    "suid_binary_enumeration": {
      "command": "find / -perm -4000 2>/dev/null",
      "output": "",
      "status": ""
    },
    "listening_services": {
      "command": "netstat -tulnp",
      "output": "",
      "status": ""
    }
  }
}

```

ifconfig wlan0 down
ifconfig wlan0 hw ether ne:wm:ac:ad:dr:es

airmon-ng check kill
iwconfig wlan0 mode monitor
ifconfig wlan0 up

deauthenticating
dont need encryption key to deauthenticate. works with any device. any network. you dont even need to connect to the device. Change MAC to client machine and request to disconnect. Change to the server MAC and tell disconnected.

airplay-ng --death [# packets] -a [network MAC] -c [target MAC] [interface]

WEP
Exploit Rc4 encryption. it’s password key plus its 24 bits initial vector is the key stream. iv is too small and written in plain text